# Bayesian Hierarchical Modeling for Stock Return Prediction

## 1. Research Objective

This notebook aims to develop and evaluate a Bayesian hierarchical model for predicting monthly S&P 500 constituent stock returns using a comprehensive set of stock-level characteristics. The primary goals are:
1.  To implement a Bayesian approach for variable selection (identifying significant predictors) and parameter estimation, building upon insights from prior LASSO and Random Forest analyses.
2.  To incorporate a hierarchical structure that groups factors by economic themes, potentially informed by external research (e.g., JKP (2023) on factor replicability).
3.  To allow for stock-specific effects (intercepts and error variances) to capture heterogeneity.
4.  To evaluate the out-of-sample predictive performance of this Bayesian model against benchmarks and simpler machine learning models using a rigorous dynamic expanding window approach.
5.  Ultimately, to assess whether the Bayesian MCMC framework can enhance prediction accuracy and provide robust insights into the drivers of stock returns.

## 2. Data

*   **Source Data:** The analysis uses the `sp500_factors_final_for_modeling.parquet` dataset. This dataset contains:
    *   Monthly observations for S&P 500 constituents.
    *   `permno` (stock identifier) and `date` (end-of-month).
    *   153 stock-level characteristics (factors), which have been imputed (cross-sectional monthly medians) and rank-standardized to the [-1,1] interval.
    *   The target variable `target_ret_t_plus_1` (the next month's realized stock return).
*   **Time Periods:**
    *   **Overall Data Period:** 1965-2023).
    *   **Initial Training Period (for model development & initial fit):** 1965-01-31 to 1989-12-31.
    *   **Validation Period (for comparing Bayesian model variants & strategies):** 1990-01-31 to 2004-12-31.
    *   **Out-of-Sample (OOS) Test Period (for final model evaluation):** 2005-01-31 to 2023-12-31.

## 3. Modeling Approach: Staged Development of Bayesian Hierarchical Model

We will develop the Bayesian model in stages using PyMC, starting with simpler specifications and iteratively adding complexity. All models will eventually be evaluated using the dynamic expanding window methodology.

### Stage 1: Simplified Bayesian Spike-and-Slab (Global Parameters)

*   **Objective:** Establish a baseline Bayesian variable selection model and ensure the MCMC machinery (PyMC setup, sampling, diagnostics) is working correctly.
*   **Model Specification:**
    *   Likelihood: \(y_{it} = \alpha + \sum_{j=1}^{P} X_{ijt} \beta_j + \epsilon_{it}\), where \(\epsilon_{it} \sim N(0, \sigma^2)\) (pooled observations, global intercept \(\alpha\), global error variance \(\sigma^2\)).
    *   Priors for \(\beta_j\):
        *   \(\gamma_j \sim \text{Bernoulli}(\pi_0)\) (Inclusion indicator for factor \(j\))
        *   \(\beta_j \sim \gamma_j N(0, \tau_1^2) + (1-\gamma_j) N(0, \tau_0^2)\) (Spike-and-Slab)
    *   Hyperpriors:
        *   \(\alpha \sim N(0, \text{sd=some_large_value})\)
        *   \(\sigma \sim \text{HalfCauchy(1)}\) (Global error standard deviation)
        *   \(\tau_0\) (spike std. dev.): Fixed to a small value (e.g., 0.01).
        *   \(\tau_1 \sim \text{HalfCauchy(1)}\) (slab std. dev., global for all selected \(\beta_j\)).
        *   \(\pi_0 \sim \text{Beta}(1, 1)\) (Global prior inclusion probability for any factor).
*   **Tasks:**
    1.  Implement in PyMC.
    2.  Fit on the initial training period (1965-1989).
    3.  Perform convergence diagnostics (trace plots, R-hat, effective sample size).
    4.  Examine Posterior Inclusion Probabilities (PIPs) for factors and posterior distributions of \(\beta_j\).
    5.  Evaluate basic predictive performance on the first year of the validation period (1990).

### Stage 2: Adding Stock-Specific Effects (Intercepts & Error Variances)

*   **Objective:** Incorporate heterogeneity across stocks as suggested by the specification PDF (\(\sigma_i^2\)).
*   **Model Specification (building on Stage 1):**
    *   Likelihood: \(y_{it} = \alpha_i + \sum_{j=1}^{P} X_{ijt} \beta_j + \epsilon_{it}\), where \(\epsilon_{it} \sim N(0, \sigma_i^2)\).
    *   Spike-and-Slab for \(\beta_j\) remains the same as Stage 1 (using global \(\pi_0, \tau_0, \tau_1\)).
    *   Priors for new parameters:
        *   Stock-specific intercepts: \(\alpha_i \sim N(\mu_{\alpha}, \sigma_{\alpha}^2)\) (Hierarchical prior)
            *   \(\mu_{\alpha} \sim N(0, \text{sd=large})\)
            *   \(\sigma_{\alpha} \sim \text{HalfCauchy(1)}\)
        *   Stock-specific error standard deviations: \(\sigma_i \sim \text{HalfCauchy(scale_sigma_i)}\) (e.g., `scale_sigma_i` could be common, or hierarchical). Alternatively, \(\sigma_i^2 \sim \text{InverseGamma}(\text{low_alpha, low_beta})\).
*   **Tasks:**
    1.  Implement in PyMC.
    2.  Fit on the initial training period (1965-1989).
    3.  Convergence diagnostics.
    4.  Compare PIPs and \(\beta_j\) posteriors with Stage 1. Assess impact of stock-specific effects.

### Stage 3: Introducing Theme-Informed Hierarchical Spike-and-Slab (Target Model)

*   **Objective:** Implement the full hierarchical model incorporating economic themes into the slab prior and using theme-specific (potentially JKP-informed) inclusion probabilities.
*   **Model Specification (building on Stage 2, aligning with User's Option 2):**
    *   Likelihood: \(y_{it} = \alpha_i + \sum_{k=1}^{K_{themes}} \sum_{j \in \text{Theme}_k} X_{ijt} \beta_{jk} + \epsilon_{it}\), where \(\epsilon_{it} \sim N(0, \sigma_i^2)\).
    *   Priors for \(\beta_{jk}\) (factor \(j\) in theme \(k\)):
        *   \(\gamma_{jk} \sim \text{Bernoulli}(\pi_k)\)
        *   \(\beta_{jk} \sim \gamma_{jk} N(\mu_k, \tau_k^2) + (1-\gamma_{jk}) N(0, \tau_0^2)\)
    *   Hyperpriors:
        *   \(\alpha_i, \sigma_i\) as in Stage 2.
        *   \(\tau_0^2\) fixed small.
        *   Theme-level mean: \(\mu_k \sim N(0, \text{sd_for_mu_k})\) (e.g., `sd_for_mu_k=1`).
        *   Theme-level std. dev. for slab: \(\tau_k \sim \text{HalfCauchy(1)}\).
        *   Theme-level prior inclusion probability: \(\pi_k \sim Beta(a_k, b_k)\), where \(a_k, b_k\) can be set based on JKP findings (e.g., higher \(a_k\)/lower \(b_k\) for more replicable themes).
*   **Tasks:**
    1.  Requires mapping factors to themes and defining JKP-informed priors for \(\pi_k\).
    2.  Implement in PyMC.
    3.  Fit on the initial training period (1965-1989).
    4.  Convergence diagnostics (will be more challenging).
    5.  Analyze PIPs (\(\gamma_{jk}\)), theme-level parameters (\(\mu_k, \tau_k, \pi_k\)), and factor coefficients \(\beta_{jk}\).

### Stage 4: Dynamic Out-of-Sample Evaluation (Validation & Test Periods)

*   **Objective:** Evaluate the chosen Bayesian model specification (likely starting with the most complex version that converges reliably from Stage 3, or a slightly simplified version if Stage 3 proves too slow/unstable for yearly refits) using the dynamic expanding window approach.
*   **Process (for chosen model):**
    1.  **Validation Period (1990-2004):**
        *   Loop yearly: define expanding training window, re-fit Bayesian model, generate predictions (posterior predictive mean) for the next year.
        *   Store predictions and actuals.
        *   Evaluate overall performance (MSE, OOS R², Portfolio Sorts, IC). Compare different Bayesian model variants if multiple were stable in earlier stages.
    2.  **Out-of-Sample Test Period (2005-2023):**
        *   Apply the best performing/most robust Bayesian model specification from the validation phase.
        *   Loop yearly: define expanding training window, re-fit Bayesian model, generate predictions.
        *   Store predictions and actuals.
        *   Evaluate final OOS performance and compare against LASSO and Random Forest benchmarks.
        *   Analyze factor significance (PIPs) over time.

## 4. Evaluation Metrics

*   **Statistical Predictive Accuracy:** Out-of-Sample R² (vs. historical mean), Mean Squared Error (MSE).
*   **Economic Significance:**
    *   Decile portfolio sorts based on predicted returns.
    *   Mean monthly returns of long-short (top decile - bottom decile) portfolios.
    *   Annualized Sharpe Ratios of long-short portfolios.
    *   (Optionally) Alphas from Fama-French factor models for L-S portfolios.
*   **Factor Importance/Selection:**
    *   For LASSO: Number and identity of selected predictors (non-zero coefficients).
    *   For Random Forest: Feature importances.
    *   For Bayesian Model: Posterior Inclusion Probabilities (PIPs) for each factor/theme; posterior distributions of coefficients.
*   **Information Coefficient (IC):** Mean monthly Spearman Rank IC between predictions and actual returns.

## 5. Software and Tools

*   **Python:** `pandas`, `numpy`, `scikit-learn`.
*   **Bayesian Modeling:** `PyMC`.
*   **Visualization:** `matplotlib`, `seaborn`.

---

In [1]:
import sys
import os

# This will print the path to the Python executable the notebook is currently using.
print(sys.executable)

c:\Users\Admin\anaconda3\envs\pymc_env\python.exe


In [2]:
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az # For MCMC diagnostics and visualization
import pytensor.tensor as pt # For more complex tensor operations if needed later
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os

In [3]:

print(f"Running on PyMC v{pm.__version__}")
print(f"Running on ArviZ v{az.__version__}")

# --- Ensure data is loaded and initial training set is prepared ---
if 'df' not in locals() or df is None:
    print("Loading data for Bayesian LASSO Stage 1...")
    processed_data_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_final_for_modeling.parquet'
    try:
        df = pd.read_parquet(processed_data_path)
        df['date'] = pd.to_datetime(df['date'])
        print(f"Successfully loaded data from {processed_data_path}. Shape: {df.shape}")

        all_columns = df.columns.tolist()
        known_identifiers_and_target = [
            'permno', 'date', 'ret','target_ret_t_plus_1' 
        ]
        non_predictor_cols = [col for col in known_identifiers_and_target if col in df.columns]
        factor_columns_base = [col for col in df.columns if col not in non_predictor_cols and df[col].dtype == 'float64']
        factor_columns = list(set(factor_columns_base))
        TARGET_COL = 'target_ret_t_plus_1'
        print(f"Identified {len(factor_columns)} predictor columns.")

        train_start_date = pd.to_datetime('1965-01-31')
        train_end_date = pd.to_datetime('1989-12-31')
        
        train_df_stage1 = df[(df['date'] >= train_start_date) & (df['date'] <= train_end_date)].copy()
        train_df_stage1.dropna(subset=[TARGET_COL] + factor_columns, inplace=True)

        if not train_df_stage1.empty:
            X_train_initial = train_df_stage1[factor_columns].values 
            y_train_initial_raw = train_df_stage1[TARGET_COL].values # Keep raw y for unscaling later if needed
            
            # Standardize y_train_initial
            y_scaler = StandardScaler()
            y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
            print(f"y_train_initial has been standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")
            
            print(f"Initial Training Set (1965-1989) prepared: X shape {X_train_initial.shape}, y shape {y_train_initial.shape}")
        else:
            print("Initial training set is empty after NaN drop. Cannot proceed.")
            X_train_initial, y_train_initial = None, None
            
    except FileNotFoundError:
        print(f"Error: File not found at {processed_data_path}.")
        X_train_initial, y_train_initial = None, None
    except Exception as e:
        print(f"An error occurred loading or preparing data: {e}")
        X_train_initial, y_train_initial = None, None
else:
    if 'X_train_initial' not in locals() or 'y_train_initial' not in locals() or X_train_initial is None:
        print("Warning: X_train_initial or y_train_initial not found. Attempting to recreate from 'df'.")
        train_df_stage1 = df[(df['date'] >= train_start_date) & (df['date'] <= train_end_date)].copy()
        train_df_stage1.dropna(subset=[TARGET_COL] + factor_columns, inplace=True)
        if not train_df_stage1.empty:
            X_train_initial = train_df_stage1[factor_columns].values
            y_train_initial_raw = train_df_stage1[TARGET_COL].values
            y_scaler = StandardScaler()
            y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
            print(f"y_train_initial has been standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")
            print(f"Initial Training Set (1965-1989) prepared: X shape {X_train_initial.shape}, y shape {y_train_initial.shape}")
        else:
            print("Initial training set is empty after NaN drop. Cannot proceed.")
            X_train_initial, y_train_initial = None, None
    else:
        if isinstance(X_train_initial, pd.DataFrame):
            X_train_initial = X_train_initial.values
        if isinstance(y_train_initial, pd.Series): # If y_train_initial was already a Series
             y_train_initial_raw = y_train_initial.values
             y_scaler = StandardScaler() # Assume it needs scaling
             y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
             print(f"Using existing y_train_initial and standardizing it. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")
        elif y_train_initial.ndim == 1 and np.abs(y_train_initial.mean()) > 1e-3 and np.abs(y_train_initial.std() - 1.0) > 1e-3 : # Check if already scaled
            print("y_train_initial appears to be a numpy array but possibly not standardized. Re-standardizing.")
            y_train_initial_raw = y_train_initial # Keep a copy if needed
            y_scaler = StandardScaler()
            y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
            print(f"y_train_initial has been standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")
        else:
            print("Using existing X_train_initial and presuming y_train_initial is already standardized if its mean is ~0 and std is ~1.")
        print(f"Shapes: X_train_initial {X_train_initial.shape}, y_train_initial {y_train_initial.shape}")



Running on PyMC v5.22.0
Running on ArviZ v0.21.0
Loading data for Bayesian LASSO Stage 1...
Successfully loaded data from C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_final_for_modeling.parquet. Shape: (344511, 156)
Identified 152 predictor columns.
y_train_initial has been standardized. Mean: -0.00, Std: 1.00
Initial Training Set (1965-1989) prepared: X shape (148018, 152), y shape (148018,)


In [18]:
import os
if X_train_initial is not None and y_train_initial is not None:
    num_predictors = X_train_initial.shape[1]
    num_observations = X_train_initial.shape[0]

    print(f"\n--- Bayesian Spike-and-Slab Model (Stage 1: Global Parameters) ---")
    print(f"Number of observations: {num_observations}, Number of predictors: {num_predictors}")

    with pm.Model() as model_stage1:
        # --- Priors ---
        # Intercept
        alpha = pm.Normal('alpha', mu=0, sigma=10) # global prior for intercept

        # Global error standard deviation
        sigma = pm.HalfCauchy('sigma', beta=1) # Weakly informative prior for error scale

        # Spike-and-Slab components for beta coefficients
        tau0 = 0.01  # Standard deviation for the spike (fixed small)
        tau1 = pm.HalfNormal('tau1', sigma=1) # Standard deviation for the slab (global)
        
        # Prior for global inclusion probability pi_0
        # pi0_hyper_a = 1.0 # Parameter for Beta prior on pi_0 (e.g. Beta(1,1) for Uniform)
        # pi0_hyper_b = 1.0 # Parameter for Beta prior on pi_0
        # pi0 = pm.Beta('pi0', alpha=pi0_hyper_a, beta=pi0_hyper_b)
        
        # Simpler: Individual pi_j for each beta, each from Beta(1,1) as per PDF discussion
        pi_j = pm.Beta('pi_j', alpha=1.0, beta=1.0, shape=num_predictors)

        # Inclusion indicators (gamma_j)
        gamma_j = pm.Bernoulli('gamma_j', p=pi_j, shape=num_predictors)

        # Coefficients beta_j
        # Define them based on whether they are in spike or slab
        # beta_j will be non-centered for better sampling if taus are very different
        beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=num_predictors)
        
        # Effective standard deviation for each beta based on gamma_j
        beta_sd = pm.math.switch(gamma_j, tau1, tau0)
        beta = pm.Deterministic('beta', beta_raw * beta_sd)
        
        # Alternative (centered) way to define beta (might sample less efficiently if tau0 is tiny):
        # beta_spike = pm.Normal('beta_spike', mu=0, sigma=tau0, shape=num_predictors)
        # beta_slab = pm.Normal('beta_slab', mu=0, sigma=tau1, shape=num_predictors)
        # beta = pm.math.switch(gamma_j, beta_slab, beta_spike, name='beta')

        # --- Likelihood ---
        # Ensure X_train_initial and y_train_initial are NumPy arrays
        mu = alpha + pm.math.dot(X_train_initial, beta)
        likelihood = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_train_initial)

        # --- MCMC Sampling ---
        print("\nStarting MCMC sampling for Stage 1 model...")
        # Start with fewer draws/tune for initial testing, then increase for final runs
        # For actual results, you'd want more draws (e.g., 2000-4000) and tune (e.g., 1000-2000)
        n_draws = 2000 
        n_tune = 1500
        n_chains = 4
        
        try:
            trace_stage1 = pm.sample(draws=n_draws, tune=n_tune, chains=n_chains, cores=min(4, os.cpu_count()), random_seed=42, target_accept=0.9) # Target_accept can be adjusted if divergences
            print("Sampling complete.")

            # --- Convergence Diagnostics & Results ---
            print("\n--- MCMC Diagnostics (Stage 1) ---")
            summary_stage1 = az.summary(trace_stage1, var_names=['alpha', 'sigma', 'tau1', 'pi_j', 'gamma_j', 'beta'])
            print(summary_stage1)

            # Check for divergences
            divergences = trace_stage1.sample_stats.diverging.sum().item()
            print(f"\nNumber of divergences: {divergences}")
            if divergences > 0:
                print("WARNING: Divergences encountered. Model may have issues. Consider reparameterization, stronger priors, or increasing target_accept.")

            # Posterior Inclusion Probabilities (PIPs)
            # For Bernoulli gamma_j, the posterior mean is the PIP
            pips = np.mean(trace_stage1.posterior['gamma_j'].values.reshape(-1, num_predictors), axis=0)
            df_pips = pd.DataFrame({'factor': factor_columns, 'pip': pips}).sort_values(by='pip', ascending=False)
            print("\nPosterior Inclusion Probabilities (PIPs) - Top 15:")
            print(df_pips.head(15))
            
            # Plot traces for a few key parameters
            # az.plot_trace(trace_stage1, var_names=['alpha', 'sigma', 'tau1'], combined=True)
            # plt.show()

        except Exception as e_sample:
            print(f"Error during MCMC sampling or analysis: {e_sample}")
            
else:
    print("Skipping Bayesian Model Stage 1 as data was not loaded/prepared.")





--- Bayesian Spike-and-Slab Model (Stage 1: Global Parameters) ---
Number of observations: 148018, Number of predictors: 152

Starting MCMC sampling for Stage 1 model...


Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [alpha, sigma, tau1, pi_j, beta_raw]
>BinaryGibbsMetropolis: [gamma_j]


Output()

Sampling 4 chains for 1_500 tune and 2_000 draw iterations (6_000 + 8_000 draws total) took 10647 seconds.
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Sampling complete.

--- MCMC Diagnostics (Stage 1) ---


c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
alpha     -0.000  0.003  -0.005    0.005      0.000    0.000   12592.0   
sigma      0.996  0.002   0.993    1.000      0.000    0.000   16228.0   
tau1       0.021  0.002   0.016    0.025      0.000    0.000    2599.0   
pi_j[0]    0.465  0.287   0.000    0.922      0.004    0.002    5713.0   
pi_j[1]    0.467  0.286   0.000    0.922      0.004    0.002    5186.0   
...          ...    ...     ...      ...        ...      ...       ...   
beta[147] -0.005  0.008  -0.020    0.009      0.000    0.000   10069.0   
beta[148] -0.007  0.006  -0.018    0.003      0.000    0.000   10201.0   
beta[149] -0.007  0.006  -0.019    0.005      0.000    0.000    8898.0   
beta[150]  0.000  0.009  -0.018    0.017      0.000    0.000    8633.0   
beta[151] -0.027  0.012  -0.049   -0.003      0.000    0.000    2519.0   

           ess_tail  r_hat  
alpha        5842.0    1.0  
sigma        5272.0    1.0  
tau1         4086.0    1

In [ ]:
full_summary = az.summary(trace_stage1)
problematic_parameters = full_summary[
    (full_summary['r_hat'] > 1.01) | 
    (full_summary['ess_bulk'] < 200) | 
    (full_summary['ess_tail'] < 200)
]
print(problematic_parameters)


In [ ]:
if 'trace_stage1' in locals():
        print("Available coordinates in the posterior group:")
        for coord_name, coord_values in trace_stage1.posterior.coords.items():
            print(f"  Coordinate Name: {coord_name}")
            # print(f"    Values: {coord_values.values[:5]}...") # Optionally print first few values
        
        # Specifically for beta, gamma_j, pi_j which have the predictor dimension
        if 'beta' in trace_stage1.posterior:
            print("\nDimensions for 'beta':")
            print(trace_stage1.posterior['beta'].dims)
        if 'gamma_j' in trace_stage1.posterior:
            print("\nDimensions for 'gamma_j':")
            print(trace_stage1.posterior['gamma_j'].dims)
        if 'pi_j' in trace_stage1.posterior:
            print("\nDimensions for 'pi_j':")
            print(trace_stage1.posterior['pi_j'].dims)

In [ ]:
if 'trace_stage1' in locals():
        problematic_var_names = ['gamma_j', 'beta', 'pi_j'] # Focus on these
        # We need to get the specific index 73 for plotting.
        # For pi_j and beta, the index will directly correspond.
        
        # Create a dictionary for coordinates if plotting specific indices of multi-dim RVs
        coords_to_plot_gamma_j = {'gamma_j_dim_0': [73]} 
        coords_to_plot_beta = {'beta_dim_0': [73]} 
        coords_to_plot_pi_j = {'pi_j_dim_0': [73]} 

        print(f"\nPlotting trace for gamma_j[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage1, var_names=['gamma_j'], coords=coords_to_plot_gamma_j, combined=True)
        plt.show()

        print(f"\nPlotting trace for beta[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage1, var_names=['beta'], coords=coords_to_plot_beta, combined=True)
        plt.show()
        
        print(f"\nPlotting trace for pi_j[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage1, var_names=['pi_j'], coords=coords_to_plot_pi_j, combined=True)
        plt.show()

**Key Discoveries from Stage 1**
    
    -The slab standard deviation tau1 is robustly estimated to be very small, implying strong shrinkage even for "included" factors
    
    -A single global tau1 is a strong assumption and might be contributing to its small estimated value. If different groups of factors (like themes) have genuinely different typical effect sizes when they are "active," a global tau1 will be a compromise, potentially too large for some and too small for others, and might settle on a smallish value if many factors have weak effects. Stage 3, with theme-specific tau_k, will be very interesting to see if it allows for more differentiated slab variances.

**STAGE 2**

In [22]:
print("Using pre-loaded 'df'. Preparing training data for Stage 2...")
train_df_stage2 = df[(df['date'] >= train_start_date) & (df['date'] <= train_end_date)].copy()
train_df_stage2.dropna(subset=[TARGET_COL] + factor_columns + ['permno'], inplace=True)
    
if not train_df_stage2.empty:
    # Ensure X_train_initial and y_train_initial are from this specific, NaN-dropped dataframe
    X_train_initial = train_df_stage2[factor_columns].values
    y_train_initial_raw = train_df_stage2[TARGET_COL].values
    if 'y_scaler' not in locals(): y_scaler = StandardScaler() # Define if not existing
    y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
    print(f"y_train_initial has been standardized for Stage 2. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")

    unique_permnos_train = train_df_stage2['permno'].unique()
    permno_to_idx_map = {permno: i for i, permno in enumerate(unique_permnos_train)}
    permno_idx_train = train_df_stage2['permno'].map(permno_to_idx_map).values
    n_stocks_train = len(unique_permnos_train)
    print(f"Initial Training Set (1965-1989) re-prepared for Stage 2: X shape {X_train_initial.shape}, y shape {y_train_initial.shape}, N_stocks: {n_stocks_train}")
else:
    print("Initial training set is empty after NaN drop. Cannot proceed.")
    X_train_initial, y_train_initial, permno_idx_train, n_stocks_train = None, None, None, 0

if X_train_initial is not None and y_train_initial is not None and n_stocks_train > 0:
    num_predictors = X_train_initial.shape[1]
    num_observations = X_train_initial.shape[0]

    print(f"\n--- Bayesian Spike-and-Slab Model (Stage 2: Stock-Specific Intercepts & Variances) ---")
    print(f"N observations: {num_observations}, N predictors: {num_predictors}, N stocks: {n_stocks_train}")

    with pm.Model() as model_stage2:
        # --- Priors for stock-specific intercepts (alpha_i) ---
        mu_alpha = pm.Normal('mu_alpha', mu=0, sigma=1) # Hyperprior for mean of alpha_i
        sigma_alpha = pm.HalfCauchy('sigma_alpha', beta=1) # Hyperprior for std dev of alpha_i
        alpha_i_offset = pm.Normal('alpha_i_offset', mu=0, sigma=1, shape=n_stocks_train)
        alpha_i = pm.Deterministic('alpha_i', mu_alpha + alpha_i_offset * sigma_alpha)

        # --- Priors for stock-specific error standard deviations (sigma_i) ---
        # Independent HalfCauchy priors for each sigma_i
        sigma_i = pm.HalfCauchy('sigma_i', beta=1, shape=n_stocks_train)
        
        # --- Priors for global beta coefficients (Spike-and-Slab) - Same as Stage 1 ---
        tau0_beta = 0.01  
        # Using HalfNormal(1) for tau1 as per last successful Stage 1 attempt
        tau1_beta = pm.HalfNormal('tau1_beta', sigma=1) 
        
        pi_j_beta = pm.Beta('pi_j_beta', alpha=1.0, beta=1.0, shape=num_predictors)
        gamma_j_beta = pm.Bernoulli('gamma_j_beta', p=pi_j_beta, shape=num_predictors)
        
        beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=num_predictors)
        beta_sd = pm.math.switch(gamma_j_beta, tau1_beta, tau0_beta)
        beta = pm.Deterministic('beta', beta_raw * beta_sd)
        
        # --- Likelihood ---
        # Mean prediction for each observation
        mean_obs = alpha_i[permno_idx_train] + pm.math.dot(X_train_initial, beta)
        # Observation-specific error standard deviation
        sigma_obs = sigma_i[permno_idx_train]
        
        likelihood = pm.Normal('y_obs', mu=mean_obs, sigma=sigma_obs, observed=y_train_initial)

        # --- MCMC Sampling ---
        print("\nStarting MCMC sampling for Stage 2 model...")
        n_draws_s2 = 2000 
        n_tune_s2 = 1500
        n_chains_s2 = 4 # Stick to 4 chains
        
        try:
            trace_stage2 = pm.sample(
                draws=n_draws_s2, 
                tune=n_tune_s2, 
                chains=n_chains_s2, 
                cores=min(4, os.cpu_count()), # Use up to 4 cores
                random_seed=123, # Change seed for different model
                init="jitter+adapt_diag",
                target_accept=0.9, 
                idata_kwargs={'log_likelihood': True}
            ) 
            print("Sampling complete for Stage 2.")

            # --- Convergence Diagnostics & Results ---
            print("\n--- MCMC Diagnostics (Stage 2) ---")
            # Summarize key global parameters and a sample of stock-specific ones / betas
            var_names_s2 = ['mu_alpha', 'sigma_alpha', 'sigma_i', 'tau1_beta', 'pi_j_beta', 'gamma_j_beta', 'beta']
            # For sigma_i and alpha_i, summary can be large. Arviz handles this by showing head/tail.
            summary_stage2 = az.summary(trace_stage2, var_names=var_names_s2) # Might be very long
            print(summary_stage2)

            divergences_s2 = trace_stage2.sample_stats.diverging.sum().item()
            print(f"\nNumber of divergences (Stage 2): {divergences_s2}")
            if divergences_s2 > 0:
                print("WARNING: Divergences encountered in Stage 2. Model may have issues.")

            # PIPs for betas
            pips_s2 = np.mean(trace_stage2.posterior['gamma_j_beta'].values.reshape(-1, num_predictors), axis=0)
            df_pips_s2 = pd.DataFrame({'factor': factor_columns, 'pip': pips_s2}).sort_values(by='pip', ascending=False)
            print("\nPosterior Inclusion Probabilities (PIPs - Stage 2) - Top 15:")
            print(df_pips_s2.head(15))
            
            print(f"\nPosterior mean for global tau1_beta (Stage 2): {trace_stage2.posterior['tau1_beta'].mean().item():.4f}")

        except Exception as e_sample:
            print(f"Error during MCMC sampling or analysis (Stage 2): {e_sample}")
else:
    print("Skipping Bayesian Model Stage 2 as data was not loaded/prepared correctly.")



Using pre-loaded 'df'. Preparing training data for Stage 2...
y_train_initial has been standardized for Stage 2. Mean: -0.00, Std: 1.00
Initial Training Set (1965-1989) re-prepared for Stage 2: X shape (148018, 152), y shape (148018,), N_stocks: 1015

--- Bayesian Spike-and-Slab Model (Stage 2: Stock-Specific Intercepts & Variances) ---
N observations: 148018, N predictors: 152, N stocks: 1015

Starting MCMC sampling for Stage 2 model...


Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [mu_alpha, sigma_alpha, alpha_i_offset, sigma_i, tau1_beta, pi_j_beta, beta_raw]
>BinaryGibbsMetropolis: [gamma_j_beta]


Output()

Sampling 4 chains for 1_500 tune and 2_000 draw iterations (6_000 + 8_000 draws total) took 13274 seconds.
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 49 divergences after tuning. Increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


Sampling complete for Stage 2.

--- MCMC Diagnostics (Stage 2) ---


c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4


              mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu_alpha     0.002  0.003  -0.003    0.006      0.000    0.000   19212.0   
sigma_alpha  0.003  0.003   0.000    0.008      0.000    0.000    4754.0   
sigma_i[0]   0.932  0.043   0.852    1.010      0.000    0.001   15221.0   
sigma_i[1]   0.769  0.085   0.621    0.934      0.001    0.001   19596.0   
sigma_i[2]   0.991  0.040   0.918    1.067      0.000    0.001   20185.0   
...            ...    ...     ...      ...        ...      ...       ...   
beta[147]    0.003  0.007  -0.011    0.017      0.000    0.000   12675.0   
beta[148]   -0.005  0.005  -0.014    0.005      0.000    0.000    9624.0   
beta[149]   -0.006  0.006  -0.016    0.005      0.000    0.000    8184.0   
beta[150]   -0.000  0.009  -0.017    0.016      0.000    0.000   10870.0   
beta[151]   -0.016  0.010  -0.036    0.003      0.000    0.000    5176.0   

             ess_tail  r_hat  
mu_alpha       6008.0    1.0  
sigma_alpha    3558.0    

In [ ]:
full_summary = az.summary(trace_stage2)
problematic_parameters = full_summary[
    (full_summary['r_hat'] > 1.01) | 
    (full_summary['ess_bulk'] < 200) | 
    (full_summary['ess_tail'] < 200)
]
print(problematic_parameters)

In [ ]:
if 'trace_stage2' in locals():
        print("Available coordinates in the posterior group:")
        for coord_name, coord_values in trace_stage2.posterior.coords.items():
            print(f"  Coordinate Name: {coord_name}")
            # print(f"    Values: {coord_values.values[:5]}...") # Optionally print first few values
        
        # Specifically for beta, gamma_j, pi_j which have the predictor dimension
        if 'beta' in trace_stage2.posterior:
            print("\nDimensions for 'beta':")
            print(trace_stage2.posterior['beta'].dims)
        if 'gamma_j' in trace_stage2.posterior:
            print("\nDimensions for 'gamma_j':")
            print(trace_stage2.posterior['gamma_j'].dims)
        if 'pi_j' in trace_stage2.posterior:
            print("\nDimensions for 'pi_j':")
            print(trace_stage2.posterior['pi_j'].dims)

In [ ]:
if 'trace_stage2' in locals():
        problematic_var_names = ['gamma_j', 'beta', 'pi_j'] # Focus on these
        # We need to get the specific index 73 for plotting.
        # For pi_j and beta, the index will directly correspond.
        
        # Create a dictionary for coordinates if plotting specific indices of multi-dim RVs
        coords_to_plot_gamma_j = {'gamma_j_beta_dim_0': [73]} 
        coords_to_plot_beta = {'beta_dim_0': [73]} 
        coords_to_plot_pi_j = {'pi_j_beta_dim_0': [73]} 

        print(f"\nPlotting trace for gamma_j[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage2, var_names=['gamma_j_beta'], coords=coords_to_plot_gamma_j, combined=True)
        plt.show()

        print(f"\nPlotting trace for beta[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage2, var_names=['beta'], coords=coords_to_plot_beta, combined=True)
        plt.show()
        
        print(f"\nPlotting trace for pi_j[73] (Factor: {factor_columns[73] if len(factor_columns)>73 else 'Unknown_Factor_73'})")
        az.plot_trace(trace_stage2, var_names=['pi_j_beta'], coords=coords_to_plot_pi_j, combined=True)
        plt.show()

**SIMPLIFIED STAGE 3**

global intercept

global standard error deviation

theme based hierarchy for beta_jk

Factor-to-Theme Mapping

In [3]:
theme_to_factors_dict = {
    'Momentum': [
        "Current price to high price over last year (prc_highprc_252d)", "Residual momentum t-12 to t-1 (resff3_12_1)",
        "Residual momentum t-6 to t-1 (resff3_6_1)", "Price momentum t-12 to t-1 (ret_12_1)",
        "Price momentum t-3 to t-1 (ret_3_1)", "Price momentum t-6 to t-1 (ret_6_1)",
        "Price momentum t-9 to t-1 (ret_9_1)", "Year 1-lagged return, nonannual (seas_1_1na)",
    ],
    'Value': [
        "Assets-to-market (at_me)", "Book-to-market equity (be_me)", "Book-to-market enterprise value (bev_mev)",
        "Net stock issues (chcsho_12m)", "Debt-to-market (debt_me)", "Dividend yield (div12m_me)", "Ebitda-to-market enterprise value (ebitda_mev)",
        "Equity duration (eq_dur)", "Net equity issuance (eqnetis_at)", "Equity net payout (eqnpo_12m)", "Net payout yield (eqnpo_me)",
        "Payout yield (eqpo_me)", "Free cash flow-to-price (fcf_me)", "Intrinsic value-to-market (ival_me)",
        "Net total issuance (netis_at)", "Earnings-to-price (ni_me)", "Operating cash flow-to-market (ocf_me)",
        "Sales-to-market (sale_me)"
    ],
    'Seasonality': [
        "Market correlation (corr_1260d)", "Coskewness (coskew_21d)", "Net debt issuance (dbnetis_at)", "Kaplan-Zingales index (kz_index)",
        "Change in long-term investments (lti_gr1a)", "Taxable income-to-book income (pi_nix)",
        "Years 11-15 lagged returns, annual (seas_11_15an)", "Years 16-20 lagged returns, annual (seas_16_20an)",
        "Years 2-5 lagged returns, annual (seas_2_5an)", "Years 6-10 lagged returns, annual (seas_6_10an)",
        "Change in short-term investments (sti_gr1a)"
    ],
    'Short-Term Reversal': [
        "Idiosyncratic skewness from the CAPM (iskew_capm_21d)", "Idiosyncratic skewness from the Fama-French 3-factor model (iskew_ff3_21d)",
        "Idiosyncratic skewness from the q-factor model (iskew_hxz4_21d)", "Short-term reversal (ret_1_0)", "Highest 5 days of return scaled by volatility (rmax5_rvol_21d)",
        "Total skewness (rskew_21d)"
    ],
    'Quality': [
        "Capital turnover (at_turnover)", "Cash-based operating profits-to-book assets (cop_at)", "Change gross margin minus change sales (dgp_dsale)",
        "Gross profits-to-assets (gp_at)", "Gross profits-to-lagged assets (gp_atl1)", "Mispricing factor: Performance (mispricing_perf)", 
        "Number of consecutive quarters with earnings increases (ni_inc8q)", "Quarterly return on assets (niq_at)", "Operating profits-to-book assets (op_at)",
        "Operating profits-to-lagged book assets (op_atl1)", "Operating leverage (opex_at)", "Quality minus Junk: Growth (qmj_growth)", 
        "Quality minus Junk: Profitability (qmj_prof)", "Quality minus Junk: Safety (qmj_safety)", "Assets turnover (sale_bev)"
        ],
    'Profit Growth': [
        "Change sales minus change Inventory (dsale_dinv)", "Change sales minus change receivables (dsale_drec)", "Change sales minus change SG&A (dsale_dsga)",
        "Change in quarterly return on assets (niq_at_chg1)", "Change in quarterly return on equity (niq_be_chg1)", "Standardized earnings surprise (niq_su)",
        "Change in operating cash flow to assets (ocf_at_chg1)", "Price momentum t-12 to t-7 (ret_12_7)", "Labor force efficiency (sale_emp_gr1)", 
        "Standardized Revenue surprise (saleq_su)", "Year 1-lagged return, annual (seas_1_1an)", "Tax expense surprise (tax_gr1a)"
    ],
    'Low Leverage': [
        "Firm age (age)", "Liquidity of market assets (aliq_mat)", "Book leverage (at_be)", "The high-low bid-ask spread (bidaskhl_21d)", "Cash-to-assets (cash_at)", 
        "Net debt-to-price (netdebt_me)", "Earnings volatility (ni_ivol)", "R+D-to-sales (rd_sale)", "R&D capital-to-book assets (rd5_at)", "Asset tangibility (tangibility)",
        "Altman Z-score (z_score)"
    ],
    'Accruals': [
        "Change in current operating working capital (cowc_gr1a)", "Operating accruals (oaccruals_at)", "Percent operating accruals (oaccruals_ni)",
        "Years 16-20 lagged returns, nonannual (seas_16_20na)", "Total accruals (taccruals_at)"
    ],
    'Debt Issuance': [
        "Abnormal corporate investment (capex_abn)", "Growth in book debt (3 years) (debt_gr3)", "Change in financial liabilities (fnl_gr1a)", "Change in noncurrent operating liabilities (ncol_gr1a)",
        "Change in net financial assets (nfna_gr1a)", "Earnings persistence (ni_ar1)", "Net operating assets (noa_at)"
    ],
    'Low Risk': [
        "Market Beta (beta_60m)", "Dimson beta (beta_dimson_21d)", "Frazzini-Pedersen market beta (betabab_1260d)", "Downside beta (betadown_252d)", "Earnings variability (earnings_variability)", "Idiosyncratic volatility from the CAPM (21 days) (ivol_capm_21d)",
        "Idiosyncratic volatility from the CAPM (252 days) (ivol_capm_252d)", "Idiosyncratic volatility from the Fama-French 3-factor model (ivol_ff3_21d)", "Idiosyncratic volatility from the q-factor model (ivol_hxz4_21d)",
        "Cash flow volatility (ocfq_saleq_std)","Maximum daily return (rmax1_21d)", "Highest 5 days of return (rmax5_21d)", "Return volatility (rvol_21d)", "Years 6-10 lagged returns, nonannual (seas_6_10na)",
        "Share turnover (turnover_126d)", "Number of zero trades with turnover as tiebreaker (6 months) (zero_trades_126d)", "Number of zero trades with turnover as tiebreaker (1 month) (zero_trades_21d)",
        "Number of zero trades with turnover as tiebreaker (12 months) (zero_trades_252d)"
    ]
}

jkp_significant_themes = ['Momentum', 'Value', 'Seasonality', 'Short-Term Reversal', 'Quality', 'Profit Growth', 'Low Leverage', 'Accruals', 'Debt Issuance', 'Low Risk']

print("Theme dictionaries defined.")

mapping_file_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/factor_table_definitions.csv'
try:
    factor_mapping_df = pd.read_csv(mapping_file_path, sep=';') # Using semicolon separator
    print(f"Successfully loaded factor definitions from {mapping_file_path}")
    
    # Create a mapping from full description to variable name
    desc_to_var_map = pd.Series(factor_mapping_df['Variable Name'].values, index=factor_mapping_df['Description']).to_dict()

    # Create the mapping from variable name to theme name
    var_to_theme_map = {}
    for theme, factor_list in theme_to_factors_dict.items():
        for factor_desc in factor_list:
            if factor_desc in desc_to_var_map:
                var_name = desc_to_var_map[factor_desc]
                var_to_theme_map[var_name] = theme
            else:
                print(f"Warning: Description '{factor_desc}' from your dictionary not found in mapping CSV.")
    
    # Create the list of theme names and get K_themes
    theme_names = sorted(list(theme_to_factors_dict.keys()))
    K_themes = len(theme_names)
    theme_to_idx = {name: i for i, name in enumerate(theme_names)}
    print(f"\nFound {K_themes} unique themes: {theme_names}")

    # Load the main processed dataframe to get the factor column order
    processed_data_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_final_for_modeling.parquet'
    df = pd.read_parquet(processed_data_path)
    
    # Re-identify factor columns from the main df to ensure correct order
    all_columns = df.columns.tolist()
    known_identifiers_and_target = ['permno', 'date', 'ret', 'target_ret_t_plus_1']
    non_predictor_cols = [col for col in known_identifiers_and_target if col in df.columns]
    factor_columns_base = [col for col in all_columns if col not in non_predictor_cols and df[col].dtype == 'float64']
    factor_columns = sorted(list(set(factor_columns_base))) # Sort for reproducibility
    num_predictors = len(factor_columns)
    print(f"Final ordered list of {num_predictors} predictor columns has been created.")
    
    # --- Corrected Logic for Handling 'Other' Theme ---
    # First, check if there are any factors that need to be categorized as 'Other'
    unmapped_factors = [var for var in factor_columns if var not in var_to_theme_map]
    if unmapped_factors:
        print(f"\nFound {len(unmapped_factors)} factors not in any specified theme. E.g.: {unmapped_factors[:5]}")
    # If the 'Other' theme doesn't already exist, add it.
        if 'Other' not in theme_names:
            print("Adding 'Other' theme to handle unmapped factors.")
            theme_names.append('Other')
            theme_to_idx['Other'] = K_themes
            K_themes += 1

    # Now that theme_to_idx is guaranteed to contain 'Other' if needed, create the final index list.
    factor_theme_indices = [theme_to_idx[var_to_theme_map.get(var, 'Other')] for var in factor_columns]
    print("\nSuccessfully created factor_theme_indices array.")

except Exception as e:
    print(f"An error occurred during the setup process: {e}")

Theme dictionaries defined.
Successfully loaded factor definitions from C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/factor_table_definitions.csv

Found 10 unique themes: ['Accruals', 'Debt Issuance', 'Low Leverage', 'Low Risk', 'Momentum', 'Profit Growth', 'Quality', 'Seasonality', 'Short-Term Reversal', 'Value']
Final ordered list of 152 predictor columns has been created.

Found 42 factors not in any specified theme. E.g.: ['aliq_at', 'ami_126d', 'at_gr1', 'be_gr1a', 'capx_gr1']
Adding 'Other' theme to handle unmapped factors.

Successfully created factor_theme_indices array.


In [4]:
# --- 1. Data and Setup for Simplified Stage 3 ---
print("--- Preparing Data for Simplified Stage 3 Model ---")

if 'factor_theme_indices' not in locals():
    print("Error: Theme mapping variables not found. Please re-run the mapping cell first.")
else:
    # --- Data Loading and Splitting ---
    processed_data_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_final_for_modeling.parquet'
    df = pd.read_parquet(processed_data_path)
    df['date'] = pd.to_datetime(df['date'])

    TARGET_COL = 'target_ret_t_plus_1'
    train_start_date = pd.to_datetime('1965-01-31')
    train_end_date = pd.to_datetime('1989-12-31')

    train_df_stage3 = df[(df['date'] >= train_start_date) & (df['date'] <= train_end_date)].copy()
    train_df_stage3.dropna(subset=[TARGET_COL] + factor_columns, inplace=True)

    X_train_initial = train_df_stage3[factor_columns].values
    y_train_initial_raw = train_df_stage3[TARGET_COL].values
    
    # Re-standardize y for this training set
    y_scaler = StandardScaler()
    y_train_initial = y_scaler.fit_transform(y_train_initial_raw.reshape(-1, 1)).flatten()
    print(f"y_train_initial has been standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")

    num_predictors = X_train_initial.shape[1]
    num_observations = X_train_initial.shape[0]

    
    pi_k_priors_a = []
    pi_k_priors_b = []
    print("\\nSetting up Beta priors for theme-level inclusion probabilities (pi_k):")
    for theme_name in theme_names:
        if theme_name in jkp_significant_themes:
            a, b = 2.0, 1.0
            print(f"  - Theme '{theme_name}': Significant (JKP) -> Beta({a}, {b})")

        else:
            a, b = 1.0, 1.0
            print(f"  - Theme '{theme_name}': Not significant -> Beta({a}, {b})")
        pi_k_priors_a.append(a)
        pi_k_priors_b.append(b)

    # --- 2. Define Simplified Stage 3 Model ---
    print(f"\\n--- Bayesian Hierarchical Model (Simplified Stage 3) ---")
    print(f"N observations: {num_observations}, N predictors: {num_predictors}, N themes: {K_themes}")

    with pm.Model() as model_stage3_simple:
        # --- Priors for Global Intercept and Variance ---
        alpha = pm.Normal('alpha', mu=0, sigma=10)
        sigma = pm.HalfCauchy('sigma', beta=1)

        # --- Hierarchical Priors for Betas based on Themes ---
        # Theme-level priors
        mu_k = pm.Normal('mu_k', mu=0, sigma=1, shape=K_themes)       # Theme-level mean
        tau_k = pm.HalfNormal('tau_k', sigma=1, shape=K_themes)      # Theme-level slab std dev
        pi_k = pm.Beta('pi_k', alpha=pi_k_priors_a, beta=pi_k_priors_b, shape=K_themes) # Theme-level inclusion prob
        
        # Spike component (global)
        tau0 = 0.01

        # --- Factor-level Priors using Theme-level parameters ---
        # Assign each factor its theme's inclusion probability
        gamma_j = pm.Bernoulli('gamma_j', p=pi_k[factor_theme_indices], shape=num_predictors)
        
        # Non-centered parameterization for beta
        beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=num_predictors)
        
        # Select mean and std dev based on gamma_j and theme
        slab_mean = mu_k[factor_theme_indices]
        slab_sd = tau_k[factor_theme_indices]
        
        effective_mean = gamma_j * slab_mean
        effective_sd = pm.math.switch(gamma_j, slab_sd, tau0)
        
        beta = pm.Deterministic('beta', effective_mean + beta_raw * effective_sd)
        
        # --- Likelihood ---
        mu_obs = alpha + pm.math.dot(X_train_initial, beta)
        likelihood = pm.Normal('y_obs', mu=mu_obs, sigma=sigma, observed=y_train_initial)

        # --- 3. MCMC Sampling ---
        print("\\nStarting MCMC sampling for Simplified Stage 3 model...")
        n_draws_s3 = 2000 
        n_tune_s3 = 1500
        n_chains_s3 = 4
        
        try:
            trace_stage3_simple = pm.sample(
                draws=n_draws_s3, 
                tune=n_tune_s3, 
                chains=n_chains_s3, 
                cores=min(4, os.cpu_count()),
                random_seed=4242, # New seed
                init="jitter+adapt_diag",
                target_accept=0.9,
                progressbar=True
            )
            print("Sampling complete for Simplified Stage 3.")
            
            # --- 4. Post-Sampling Analysis ---
            print("\\n--- MCMC Diagnostics (Simplified Stage 3) ---")
            var_names_s3 = ['alpha', 'sigma', 'mu_k', 'tau_k', 'pi_k', 'gamma_j', 'beta']
            summary_stage3 = az.summary(trace_stage3_simple, var_names=var_names_s3, coords={'pi_k_dim_0': theme_names})
            print(summary_stage3)

            divergences_s3 = trace_stage3_simple.sample_stats.diverging.sum().item()
            print(f"\\nNumber of divergences: {divergences_s3}")
            if divergences_s3 > 0:
                print("WARNING: Divergences encountered.")
            
            # PIPs
            pips_s3 = np.mean(trace_stage3_simple.posterior['gamma_j'].values.reshape(-1, num_predictors), axis=0)
            df_pips_s3 = pd.DataFrame({'factor': factor_columns, 'pip': pips_s3}).sort_values(by='pip', ascending=False)
            print("\\nPosterior Inclusion Probabilities (PIPs) - Top 15:")
            print(df_pips_s3.head(15))

        except Exception as e:
            print(f"An error occurred during MCMC sampling or analysis: {e}")

--- Preparing Data for Simplified Stage 3 Model ---
y_train_initial has been standardized. Mean: -0.00, Std: 1.00
\nSetting up Beta priors for theme-level inclusion probabilities (pi_k):
  - Theme 'Accruals': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Debt Issuance': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Low Leverage': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Low Risk': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Momentum': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Profit Growth': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Quality': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Seasonality': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Short-Term Reversal': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Value': Significant (JKP) -> Beta(2.0, 1.0)
  - Theme 'Other': Not significant -> Beta(1.0, 1.0)
\n--- Bayesian Hierarchical Model (Simplified Stage 3) ---
N observations: 148018, N predictors: 152, N themes: 11
\nStarting MCMC sampling for Simplified S

Multiprocess sampling (4 chains in 4 jobs)
CompoundStep
>NUTS: [alpha, sigma, mu_k, tau_k, pi_k, beta_raw]
>BinaryGibbsMetropolis: [gamma_j]


Output()

Sampling 4 chains for 1_500 tune and 2_000 draw iterations (6_000 + 8_000 draws total) took 76275 seconds.
c:\Users\Admin\anaconda3\envs\pymc_env\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
There were 1042 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


Sampling complete for Simplified Stage 3.
\n--- MCMC Diagnostics (Simplified Stage 3) ---
An error occurred during MCMC sampling or analysis: 'Coords should follow mapping format {coord_name:[dim1, dim2]}. Check that coords structure is correct and dimensions are valid. "not all values found in index \'pi_k_dim_0\'"'


In [6]:
# Assuming 'trace_stage3_simple' holds the results from your failed run.
if 'trace_stage3_simple' in locals():
    print("--- Diagnosing the failed Stage 3 trace ---")
    
    try:
        # First, let's get a summary without trying to apply fancy coordinates
        # This will help us see the raw R-hat and ESS values.
        full_summary_s3 = az.summary(trace_stage3_simple)

        print("\nSummary of parameters with convergence issues (R-hat > 1.05 or ESS < 100):")
        
        problematic_parameters = full_summary_s3[
            (full_summary_s3['r_hat'] > 1.01) | 
            (full_summary_s3['ess_bulk'] < 100) | 
            (full_summary_s3['ess_tail'] < 100)
        ]
        
        # Displaying with more rows to see all potential issues
        with pd.option_context('display.max_rows', 200):
            print(problematic_parameters)

    except Exception as e:
        print(f"An error occurred during diagnostics: {e}")
else:
    print("The trace object 'trace_stage3_simple' was not found. Please ensure the previous cell was run.")


--- Diagnosing the failed Stage 3 trace ---


c:\Users\Admin\anaconda3\envs\pymc_env\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\envs\pymc_env\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4
c:\Users\Admin\anaconda3\envs\pymc_env\Lib\site-packages\arviz\stats\diagnostics.py:596: RuntimeWarning: invalid value encountered in scalar divide
  (between_chain_variance / within_chain_variance + num_samples - 1) / (num_samples)
c:\Users\Admin\anaconda3\envs\pymc_env\Lib\site-packages\arviz\stats\diagnostics.py:991: RuntimeWarning: invalid value encountered in scalar divide
  varsd = varvar / evar / 4



Summary of parameters with convergence issues (R-hat > 1.05 or ESS < 100):
                mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  \
mu_k[2]       -0.005  0.943  -1.789    1.763      0.016    0.013    3731.0   
mu_k[3]       -0.021  0.051  -0.117    0.039      0.003    0.011     361.0   
mu_k[6]        0.094  0.900  -1.734    1.766      0.046    0.084     353.0   
mu_k[7]        0.004  0.386  -0.956    0.651      0.005    0.159     156.0   
gamma_j[1]     0.026  0.160   0.000    0.000      0.023    0.068      49.0   
gamma_j[3]     0.027  0.161   0.000    0.000      0.024    0.069      47.0   
gamma_j[4]     0.006  0.077   0.000    0.000      0.005    0.030     272.0   
gamma_j[5]     0.012  0.109   0.000    0.000      0.008    0.036     188.0   
gamma_j[6]     0.040  0.195   0.000    0.000      0.047    0.110      18.0   
gamma_j[8]     0.106  0.308   0.000    1.000      0.035    0.045      76.0   
gamma_j[9]     0.295  0.456   0.000    1.000      0.057    0.026  

In [ ]:
# --- 1. Data and Setup for Optimized Stage 3 Model ---
print("--- Preparing Data for Optimized Stage 3 Model ---")

if 'factor_theme_indices' not in locals():
    print("Error: Theme mapping variables not found. Please re-run the mapping cell first.")
else:
    # This data setup is correct and can be reused.
    # We just need to ensure the variables are available for the model cell.
    print(f"Using existing data: {X_train_initial.shape[0]} observations, {X_train_initial.shape[1]} predictors.")
    print(f"y_train_initial is standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")


# --- 2. Define Optimized Stage 3 Model (Non-Centered Parameterization) ---
print(f"\n--- Bayesian Hierarchical Model (Optimized Stage 3 with Non-Centered Priors) ---")
print(f"N observations: {num_observations}, N predictors: {num_predictors}, N themes: {K_themes}")

with pm.Model() as model_stage3_optimized:
    # --- Priors for Global Intercept and Variance (These are not hierarchical, so they are fine) ---
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    sigma = pm.HalfCauchy('sigma', beta=1)

    # --- Hierarchical Priors for Betas using NON-CENTERED parameterization ---
    # --- Theme-level priors ---
    # We define offsets from a base distribution (e.g., N(0,1))
    mu_k_offset = pm.Normal('mu_k_offset', mu=0, sigma=1, shape=K_themes)
    # The scale for the hyperprior on mu_k can be set here. 1 is a reasonable default.
    mu_k = pm.Deterministic('mu_k', 0 + mu_k_offset * 1.0) # mu_k = hyper_mean + offset * hyper_sd

    tau_k_offset = pm.HalfNormal('tau_k_offset', sigma=1, shape=K_themes)
    # The scale for the hyperprior on tau_k can be set here.
    tau_k = pm.Deterministic('tau_k', 0 + tau_k_offset * 1.0) # tau_k = hyper_mean + offset * hyper_sd

    # --- Theme-level INCLUSION PROBABILITY (p_k) ---
    # This structure is generally stable and does not need reparameterization.
    pi_k = pm.Beta('pi_k', alpha=pi_k_priors_a, beta=pi_k_priors_b, shape=K_themes) 
    
    # --- Spike component (global) ---
    tau0 = 0.01

    # --- Factor-level Priors (gamma_j and beta) ---
    # Inclusion indicator for each factor, determined by its theme's inclusion probability
    gamma_j = pm.Bernoulli('gamma_j', p=pi_k[factor_theme_indices], shape=num_predictors)
    
    # NON-CENTERED parameterization for beta
    # Sampler works on beta_raw, which is a simple N(0,1)
    beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=num_predictors)
    
    # Determine the mean and std dev for each beta based on its theme and inclusion status
    slab_mean = mu_k[factor_theme_indices]
    slab_sd = tau_k[factor_theme_indices]
    
    # Use pm.math.switch to select mean and sd based on gamma_j
    effective_mean = pm.math.switch(gamma_j, slab_mean, 0) # if in slab use theme mean, else 0
    effective_sd = pm.math.switch(gamma_j, slab_sd, tau0)   # if in slab use theme sd, else spike sd
    
    # Deterministically calculate the final beta. The sampler doesn't see this directly.
    beta = pm.Deterministic('beta', effective_mean + beta_raw * effective_sd)
    
    # --- Likelihood ---
    mu_obs = alpha + pm.math.dot(X_train_initial, beta)
    likelihood = pm.Normal('y_obs', mu=mu_obs, sigma=sigma, observed=y_train_initial)

    # --- 3. MCMC Sampling ---
    print("\nStarting MCMC sampling for Optimized Stage 3 model...")
    n_draws_s3_opt = 2000 
    n_tune_s3_opt = 1500
    n_chains_s3_opt = 6
    
    try:
        # Save the trace to a file to avoid re-running if analysis fails
        trace_filename_s3 = "trace_stage3_optimized.nc"
        if not os.path.exists(trace_filename_s3):
            trace_stage3_optimized = pm.sample(
                draws=n_draws_s3_opt, 
                tune=n_tune_s3_opt, 
                chains=n_chains_s3_opt, 
                cores= 6,
                random_seed=5678, # New seed for new model
                init="jitter+adapt_diag",
                target_accept=0.9, # Start with 0.9, can increase to 0.95 if divergences persist
                progressbar=True,
                idata_kwargs={'log_likelihood': True}
            )
            print(f"Sampling complete. Saving trace to {trace_filename_s3}")
            trace_stage3_optimized.to_netcdf(trace_filename_s3)
        else:
            print(f"Loading existing trace from {trace_filename_s3}")
            trace_stage3_optimized = az.from_netcdf(trace_filename_s3)

        # --- 4. Post-Sampling Analysis ---
        print("\n--- MCMC Diagnostics (Optimized Stage 3) ---")
        
        # Define vars to summarize
        vars_to_summarize = ['alpha', 'sigma', 'mu_k', 'tau_k', 'pi_k']
        coords_for_summary = {'pi_k_dim_0': theme_names, 'mu_k_dim_0': theme_names, 'tau_k_dim_0': theme_names}
        
        summary_stage3_opt = az.summary(trace_stage3_optimized, var_names=vars_to_summarize, coords=coords_for_summary)
        print(summary_stage3_opt)

        divergences_s3_opt = trace_stage3_optimized.sample_stats.diverging.sum().item()
        print(f"\nNumber of divergences: {divergences_s3_opt}")
        if divergences_s3_opt > 0:
            print("WARNING: Divergences encountered. Consider increasing target_accept to 0.95 or checking priors.")
        
        pips_s3_opt = np.mean(trace_stage3_optimized.posterior['gamma_j'].values.reshape(-1, num_predictors), axis=0)
        df_pips_s3_opt = pd.DataFrame({'factor': factor_columns, 'pip': pips_s3_opt}).sort_values(by='pip', ascending=False)
        
        print("\nPosterior Inclusion Probabilities (PIPs) - Top 15:")
        print(df_pips_s3_opt.head(15))

    except Exception as e:
        print(f"An error occurred during MCMC sampling or analysis: {e}")

--- Preparing Data for Optimized Stage 3 Model ---
Using existing data: 148018 observations, 152 predictors.
y_train_initial is standardized. Mean: -0.00, Std: 1.00

--- Bayesian Hierarchical Model (Optimized Stage 3 with Non-Centered Priors) ---
N observations: 148018, N predictors: 152, N themes: 11

Starting MCMC sampling for Optimized Stage 3 model...


Multiprocess sampling (6 chains in 6 jobs)
CompoundStep
>NUTS: [alpha, sigma, mu_k_offset, tau_k_offset, pi_k, beta_raw]
>BinaryGibbsMetropolis: [gamma_j]


Output()

In [ ]:
# --- 1. Data and Setup for Semi-Hierarchical Model ---
print("--- Preparing Data for Semi-Hierarchical Model ---")

if 'factor_theme_indices' not in locals():
    print("Error: Theme mapping variables not found. Please re-run the mapping cell first.")
else:
    # This data setup is correct and can be reused.
    print(f"Using existing data: {X_train_initial.shape[0]} observations, {X_train_initial.shape[1]} predictors.")
    print(f"y_train_initial is standardized. Mean: {y_train_initial.mean():.2f}, Std: {y_train_initial.std():.2f}")


# --- 2. Define Semi-Hierarchical Model (Global Mean Slab) ---
print(f"\n--- Bayesian Semi-Hierarchical Model (Theme-level tau_k and pi_k, Global mu_slab) ---")
print(f"N observations: {num_observations}, N predictors: {num_predictors}, N themes: {K_themes}")

with pm.Model() as model_stage3_semi_hierarchical:
    # --- Priors for Global Intercept and Variance ---
    alpha = pm.Normal('alpha', mu=0, sigma=10)
    sigma = pm.HalfCauchy('sigma', beta=1)

    # --- Hierarchical Priors (Non-Centered) ---
    
    # Slab Mean: GLOBAL, not per-theme
    mu_slab = pm.Normal('mu_slab', mu=0, sigma=1)

    # Slab Std Dev: PER-THEME (Hierarchical)
    tau_k_offset = pm.HalfNormal('tau_k_offset', sigma=1, shape=K_themes)
    tau_k = pm.Deterministic('tau_k', 0 + tau_k_offset * 1.0) 

    # Inclusion Probability: PER-THEME (Hierarchical)
    pi_k = pm.Beta('pi_k', alpha=pi_k_priors_a, beta=pi_k_priors_b, shape=K_themes)
    
    # --- Spike Component ---
    tau0 = 0.01

    # --- Factor-level Priors ---
    gamma_j = pm.Bernoulli('gamma_j', p=pi_k[factor_theme_indices], shape=num_predictors)
    
    # Non-centered parameterization for beta
    beta_raw = pm.Normal('beta_raw', mu=0, sigma=1, shape=num_predictors)
    
    # Select std dev based on gamma_j and theme, but mean is now global for the slab
    slab_sd = tau_k[factor_theme_indices]
    
    effective_mean = pm.math.switch(gamma_j, mu_slab, 0) 
    effective_sd = pm.math.switch(gamma_j, slab_sd, tau0)
    
    beta = pm.Deterministic('beta', effective_mean + beta_raw * effective_sd)
    
    # --- Likelihood ---
    mu_obs = alpha + pm.math.dot(X_train_initial, beta)
    likelihood = pm.Normal('y_obs', mu=mu_obs, sigma=sigma, observed=y_train_initial)

    # --- 3. MCMC Sampling ---
    print("\nStarting MCMC sampling for Semi-Hierarchical model...")
    n_draws_s3_semi = 2000 
    n_tune_s3_semi = 1500
    n_chains_s3_semi = 6
    
    try:
        trace_filename_s3_semi = "trace_stage3_semi_hierarchical.nc"
        if not os.path.exists(trace_filename_s3_semi):
            trace_stage3_semi = pm.sample(
                draws=n_draws_s3_semi, 
                tune=n_tune_s3_semi, 
                chains=n_chains_s3_semi, 
                cores=6,
                random_seed=6789,
                init="jitter+adapt_diag",
                target_accept=0.95, # Start with a higher target_accept as a precaution
                progressbar=True,
                idata_kwargs={'log_likelihood': True}
            )
            print(f"Sampling complete. Saving trace to {trace_filename_s3_semi}")
            trace_stage3_semi.to_netcdf(trace_filename_s3_semi)
        else:
            print(f"Loading existing trace from {trace_filename_s3_semi}")
            trace_stage3_semi = az.from_netcdf(trace_filename_s3_semi)

        # --- 4. Post-Sampling Analysis ---
        print("\n--- MCMC Diagnostics (Semi-Hierarchical Model) ---")
        vars_to_summarize = ['alpha', 'sigma', 'mu_slab', 'tau_k', 'pi_k']
        coords_for_summary = {'pi_k_dim_0': theme_names, 'tau_k_dim_0': theme_names}
        
        summary_stage3_semi = az.summary(trace_stage3_semi, var_names=vars_to_summarize, coords=coords_for_summary)
        print(summary_stage3_semi)

        divergences_s3_semi = trace_stage3_semi.sample_stats.diverging.sum().item()
        print(f"\nNumber of divergences: {divergences_s3_semi}")
        if divergences_s3_semi > 0:
            print("WARNING: Divergences encountered.")
        
        pips_s3_semi = np.mean(trace_stage3_semi.posterior['gamma_j'].values.reshape(-1, num_predictors), axis=0)
        df_pips_s3_semi = pd.DataFrame({'factor': factor_columns, 'pip': pips_s3_semi}).sort_values(by='pip', ascending=False)
        
        print("\nPosterior Inclusion Probabilities (PIPs) - Top 15:")
        print(df_pips_s3_semi.head(15))

    except Exception as e:
        print(f"An error occurred during MCMC sampling or analysis: {e}")